# <font style="font-family:roboto;color:#455e6c"> Calculation of phase diagrams </font>  

<div class="admonition note" name="html-admonition" style="background:#e3f2fd; padding: 10px">
<font style="font-family:roboto;color:#455e6c"> <b> DPG Tutorial: Automated Workflows and Machine Learning for Materials Science Simulations </b> </font> </br>
<font style="font-family:roboto;color:#455e6c"> 8 March 2026 </font> </br> </br>
Sriram Anand, Prabhath Chilakalapudi, Haitham Gaafer, Marvin Poul, Jan Janssen, Jörg Neugebauer </br>
<i> Max Planck Institute for Sustainable Materials </i></br>
</br>
Sarath Menon, Minaam Qamar, Ralf Drautz </br>
<i> Ruhr-Universität Bochum </i></br>
</br>
Tilmann Hickel </br>
<i> Bundesanstalt für Materialforschung und -prüfung </i></br>
</div>

## <font style="font-family:roboto;color:#455e6c"> Calculate free energy with temperature for solid</font> 

By now we have seen how the GUI works. We try to calculate the free energy of an Al FCC structure.

Helpful nodes:
- `atomistic -> structure -> build -> Bulk` to create a structure. Make sure to check `cubic` box.
- `atomistic -> structure -> transform -> Repeat` to create a supercell. A 3x3x3 cell should work for this example.
- `atomistic -> thermodynamics -> calphy -> InputClass` to set inputs for free energy calculation.
- `atomistic -> thermodynamics -> calphy -> SolidFreeEnergyWithTemp` for calculating the free energy. A good potential which is fast to try out this calculation would be `2009--Mendelev-M-I--Al-Mg--LAMMPS--ipr1`
- `atomistic -> thermodynamics -> calphy -> PlotFreeEnergy` for a plot.

In [1]:
from core import Workflow
from core.gui import PyironFlow

from pyiron_nodes.atomistic.structure.build import Bulk
from pyiron_nodes.atomistic.structure.transform import Repeat
from pyiron_nodes.atomistic.calculator.calphy import (
    InputClass,
    SolidFreeEnergyWithTemp,
    PlotFreeEnergy
)

In [2]:
wf_free_energy_unary = Workflow("Free_Energy_Unary_Solution")

In [3]:
wf_free_energy_unary.BulkStructure = Bulk(name="Mg", crystalstructure="hcp", a=3.21, c=5.21, cubic=False)
wf_free_energy_unary.RepeatStructure = Repeat(structure=wf_free_energy_unary.BulkStructure, repeat_scalar=3)
wf_free_energy_unary.CalphyInputClass = InputClass()
wf_free_energy_unary.SolidFreeEnergyWithTemp = SolidFreeEnergyWithTemp(
    input_class=wf_free_energy_unary.CalphyInputClass,
    structure=wf_free_energy_unary.RepeatStructure,
    potential="2009--Mendelev-M-I--Al-Mg--LAMMPS--ipr1",
    delete_folder=False,
)
wf_free_energy_unary.PlotUnaryFreeEnergy = PlotFreeEnergy(
    temperature=wf_free_energy_unary.SolidFreeEnergyWithTemp.outputs.temperature, 
    free_energy=wf_free_energy_unary.SolidFreeEnergyWithTemp.outputs.free_energy
)

In [4]:
workflow_path = "pyiron_core_data/workflows"
pf_free_energy_unary = PyironFlow(
    wf_list=[wf_free_energy_unary],
    nodes_path="pyiron_nodes", 
    workflow_path=workflow_path
)
pf_free_energy_unary.gui

## <font style="font-family:roboto;color:#455e6c"> Calculate melting temperature </font> 

Now we take a step forward and calculate the melting temperature. The melting temperature is the temperature at which the free energies of the solid and liquid phases are equal. 

In addition to the nodes above, the following would be useful:
- The steps for solid free energy are same as above, but with good temperature window to be set in the `InputClass` would be 800-1100 K.
- For liquid, we need to create a structure first. There are two ways to do it:
    - either check the `melting_cycle` box in `InputClass`
    - or use `atomistic -> structure -> transform -> Rattle` with `stdev` of approximately 0.6 to create a structure, similar to liquid. Hint: Use the `Plot3D` node to see how it looks like!
- `atomistic -> thermodynamics -> calphy -> LiquidFreeEnergyWithTemp` for calculating the free energy of the liquid phase
- A good temperature window to be set in the `InputClass` would be 800-1100 K.

In [5]:
from core import Workflow
from core.gui import PyironFlow

from pyiron_nodes.atomistic.structure.build import Bulk
from pyiron_nodes.atomistic.structure.transform import Repeat, Rattle
from pyiron_nodes.atomistic.calculator.calphy import (
    InputClass,
    SolidFreeEnergyWithTemp,
    LiquidFreeEnergyWithTemp
)
from pyiron_nodes.thermodynamics.landau import (
    TemperatureDependentLinePhase,
    TransitionTemperature
)

In [6]:
wf_transition_temp_unary = Workflow("Transition_Temp_Unary_Solution")

In [7]:
wf_transition_temp_unary.BulkStructure = Bulk(name="Mg", crystalstructure="hcp", a=3.21, c=5.21, cubic=False)
wf_transition_temp_unary.RepeatStructure = Repeat(structure=wf_transition_temp_unary.BulkStructure, repeat_scalar=3)
wf_transition_temp_unary.RattleStructure = Rattle(structure=wf_transition_temp_unary.RepeatStructure, seed=42, stdev=0.6)
wf_transition_temp_unary.CalphyInputClass = InputClass(temperature=900, temperature_stop=1100)

wf_transition_temp_unary.SolidFreeEnergyWithTemp = SolidFreeEnergyWithTemp(
    input_class=wf_transition_temp_unary.CalphyInputClass,
    structure=wf_transition_temp_unary.RepeatStructure, 
    potential="2009--Mendelev-M-I--Al-Mg--LAMMPS--ipr1",
)
wf_transition_temp_unary.LiquidFreeEnergyWithTemp = LiquidFreeEnergyWithTemp(
    input_class=wf_transition_temp_unary.CalphyInputClass,
    structure=wf_transition_temp_unary.RattleStructure,
    potential="2009--Mendelev-M-I--Al-Mg--LAMMPS--ipr1",
)

wf_transition_temp_unary.SolidLinePhase = TemperatureDependentLinePhase(
    name="MgSolid", 
    fixed_concentration=0,
    temperatures=wf_transition_temp_unary.SolidFreeEnergyWithTemp.outputs.temperature, 
    free_energies=wf_transition_temp_unary.SolidFreeEnergyWithTemp.outputs.free_energy
)
wf_transition_temp_unary.LiquidLinePhase = TemperatureDependentLinePhase(
    name="MgLiquid",
    fixed_concentration=0,
    temperatures=wf_transition_temp_unary.LiquidFreeEnergyWithTemp.outputs.temperature,
    free_energies=wf_transition_temp_unary.LiquidFreeEnergyWithTemp.outputs.free_energy
)
wf_transition_temp_unary.TransitionTemperature = TransitionTemperature(
    phase1=wf_transition_temp_unary.SolidLinePhase, 
    phase2=wf_transition_temp_unary.LiquidLinePhase, 
    Tmin=800, 
    Tmax=1100
)

In [8]:
workflow_path = "pyiron_core_data/workflows"
pf_transition_temp_unary = PyironFlow(
    wf_list=[wf_transition_temp_unary], 
    nodes_path="pyiron_nodes", 
    workflow_path=workflow_path
)

pf_transition_temp_unary.gui

## <font style="font-family:roboto;color:#455e6c"> Calculating a binary phase diagram </font> 

Now that we have seen the thermodynamic functions in Landau, we will make a simple eutectic binary phase diagram with Landau.

In [9]:
from core import Workflow
from core.gui import PyironFlow

from pyiron_nodes.thermodynamics.landau import (
    LinePhase, 
    IdealSolution, 
    TemperatureDependentLinePhase, 
    CalcPhaseDiagram, 
    PlotConcPhaseDiagram
)
from pyiron_nodes.basic.math import Linspace
from pyiron_nodes.basic.list import List5


In [10]:
wf_phase_diagram_binary = Workflow("Phase_Diagram_Binary_Solution")

In [11]:
wf_phase_diagram_binary.SolidLinePhaseA = LinePhase(name="SolidA", fixed_concentration=0, line_energy=2.45, line_entropy=0.00001)
wf_phase_diagram_binary.SolidLinePhaseB = LinePhase(name="SolidB", fixed_concentration=1, line_energy=2.95, line_entropy=0.00001)
wf_phase_diagram_binary.LiquidLinePhaseA = LinePhase(name="LiquidA", fixed_concentration=0, line_energy=2.5, line_entropy=0.0001)
wf_phase_diagram_binary.LiquidLinePhaseB = LinePhase(name="LiquidB", fixed_concentration=1, line_energy=3.1, line_entropy=0.0002)
wf_phase_diagram_binary.IdealSolution = IdealSolution(
    name="liquid", 
    phase1=wf_phase_diagram_binary.LiquidLinePhaseA, 
    phase2=wf_phase_diagram_binary.LiquidLinePhaseB
)
wf_phase_diagram_binary.TemperaturesLinspace = Linspace(
    x_min=250,
    x_max=1000,
    num_points=25
)
wf_phase_diagram_binary.PhasesList = List5(
    x1=wf_phase_diagram_binary.SolidLinePhaseA,
    x2=wf_phase_diagram_binary.SolidLinePhaseB,
    x3=wf_phase_diagram_binary.IdealSolution
)
wf_phase_diagram_binary.CalcPhaseDiagram = CalcPhaseDiagram(
    phases=wf_phase_diagram_binary.PhasesList,
    temperatures=wf_phase_diagram_binary.TemperaturesLinspace,
    chemical_potentials=50
)
wf_phase_diagram_binary.PlotConcPhaseDiagram = PlotConcPhaseDiagram(
    phase_data=wf_phase_diagram_binary.CalcPhaseDiagram,
    plot_tielines=True
)

In [12]:
workflow_path = "pyiron_core_data/workflows"
pf_phase_diagram_binary = PyironFlow(
    wf_list=[wf_phase_diagram_binary], 
    nodes_path="pyiron_nodes", 
    workflow_path=workflow_path
)

pf_phase_diagram_binary.gui

## <font style="font-family:roboto;color:#455e6c"> MgCa phase diagram </font> 

Now we will take the same approach to calculate a phase diagram with real data, in this case the MgCa system. The data can be found [here](https://github.com/eisenforschung/mgalca-mtp-data).

In [13]:
from core import Workflow
from core.gui import PyironFlow

from pyiron_nodes.basic.file import ReadDataFrame
from pyiron_nodes.basic.math import Linspace
from pyiron_nodes.thermodynamics.landau import (
    PhasesFromDataFrame, 
    CalcPhaseDiagram, 
    PlotConcPhaseDiagram, 
    PlotMuPhaseDiagram
)

In [14]:
wf_phase_diagram_MgCa = Workflow("Phase_Diagram_MgCa_Solution")

In [15]:
wf_phase_diagram_MgCa.ReadMgCaDataFrame = ReadDataFrame(filename="data/MgCaFreeEnergies.pckl.gz", compression="gzip")
wf_phase_diagram_MgCa.PhasesFromDataFrame = PhasesFromDataFrame(dataframe=wf_phase_diagram_MgCa.ReadMgCaDataFrame)
wf_phase_diagram_MgCa.TemperaturesLinspace = Linspace(x_min=200, x_max=1200, num_points=50)
wf_phase_diagram_MgCa.CalculatePhaseDiagram = CalcPhaseDiagram(
    phases=wf_phase_diagram_MgCa.PhasesFromDataFrame.outputs.phase_list,
    temperatures=wf_phase_diagram_MgCa.TemperaturesLinspace,
    chemical_potentials=50
)
wf_phase_diagram_MgCa.PlotConcentrationPhaseDiagram = PlotConcPhaseDiagram(phase_data=wf_phase_diagram_MgCa.CalculatePhaseDiagram, plot_tielines=True)
wf_phase_diagram_MgCa.PlotMuPhaseDiagram = PlotMuPhaseDiagram(phase_data=wf_phase_diagram_MgCa.CalculatePhaseDiagram)

In [16]:
workflow_path = "pyiron_core_data/workflows"
pf_phase_diagram_MgCa = PyironFlow(
    wf_list=[wf_phase_diagram_MgCa], 
    nodes_path="pyiron_nodes", 
    workflow_path=workflow_path
)

pf_phase_diagram_MgCa.gui

---
---

### <font style="font-family:roboto;color:#455e6c"> Software used in this notebook </font>  

- [calphy](https://calphy.org)
- [landau](https://github.com/eisenforschung/landau)
- [LAMMPS](https://www.lammps.org/)
- [pyiron](https://pyiron.org/)
- [pyiron_workflow](https://github.com/pyiron/pyiron_workflow)

<img src="img/logo_roll.png" width="1200">